# Tutorial 07: Asynchronous and Long-Running A2A (Offline-First)

Focused tutorial for `A2A/07_Asynchronous _Long-Running_A2A`, covering deferred execution, task IDs, and polling-based completion.

## 1) Where We Are in the Journey

`06_Multi-Agent_Workflow` handled synchronous step-by-step orchestration.
This step exists because real tasks may require waiting on external systems.
We need submission, status tracking, and delayed result retrieval.

## 2) What You Will Learn

- Submit/status async interaction model
- Why task IDs decouple request from completion
- Polling as a simple long-running coordination pattern
- How this maps to A2A control flow + MCP execution

## 3) Why This Matters

Long-running operations are common in production workflows.
Without async control primitives, controllers block and fail to scale.
This folder introduces deferred execution mechanics essential for resilient orchestration.

## 4) Core Concept Deep Dive

Core idea: immediate acknowledgement + delayed completion.
Pattern: `submit` returns task id, then client polls `status` until terminal state.
Trade-off: polling is simple and universal, but less efficient than push callbacks or streams.
Notebook examples in this folder call `/submit` then `/status/{task_id}`.

## 5) Code Walkthrough (Only `A2A/07_Asynchronous _Long-Running_A2A`)

- Architecture notes outline planner/router/executor loop in long-running mode.
- Code cells post to `http://localhost:9000/submit` with endpoint + payload.
- Follow-up cell polls `http://localhost:9000/status/<task_id>`.
This tutorial keeps same protocol shape while simulating server behavior offline.

In [ ]:
import time
import uuid

TASKS = {}

def submit_job(endpoint, payload):
    task_id = str(uuid.uuid4())
    TASKS[task_id] = {'status': 'queued', 'endpoint': endpoint, 'payload': payload, 'created_at': time.time(), 'result': None}
    return {'task_id': task_id, 'status': 'accepted'}

def process_tick(task_id):
    t = TASKS[task_id]
    age = time.time() - t['created_at']
    if age < 1:
        t['status'] = 'running'
    else:
        a = t['payload']['args']
        t['result'] = a['a'] + a['b']
        t['status'] = 'completed'

def get_status(task_id):
    if task_id not in TASKS:
        return {'status': 'not_found'}
    process_tick(task_id)
    t = TASKS[task_id]
    return {'status': t['status'], 'result': t['result']}

In [ ]:
submit_res = submit_job('http://localhost:8000/invoke', {'tool': 'add', 'args': {'a': 2, 'b': 3}})
print('SUBMIT:', submit_res)
task_id = submit_res['task_id']

while True:
    s = get_status(task_id)
    print('STATUS:', s)
    if s['status'] == 'completed':
        break
    time.sleep(0.5)

In [ ]:
print('FINAL RESULT:', get_status(task_id)['result'])

## 6) System Flow Explanation

1. Client submits job and gets task ID.
2. Controller schedules work asynchronously.
3. Client polls status endpoint.
4. Task transitions queued -> running -> completed.
5. Final result is fetched when terminal state is reached.

## 7) Important Concepts You Might Miss

- Task ID is the durable correlation key.
- Submission success does not mean execution success.
- Terminal states should be explicit (`completed`, `failed`, `rejected`).

## 8) Common Mistakes / Pitfalls

- Blocking client while pretending to be async.
- No timeout policy for polling loops.
- Losing task state after process restart.
- Returning incomplete result structures.

## 9) Key Takeaways

- Async orchestration needs submit/status semantics.
- Polling is baseline, not final architecture.
- Long-running support is essential for real multi-agent systems.

## 10) What’s Next (Strict Preview)

`08_streaming_a2a` focuses on incremental output delivery while execution is in progress.
It solves how to surface partial progress instead of waiting for one final response.
This matters for user experience and observability in long-running workflows.